# Documentação — 08 - Ingestão Silver Unificada: Limpeza, Tratamento e Validação de Regras
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** `ecommerce_itens_pedido`, `ecommerce_produtos`, `ecommerce_categorias`, `ecommerce_rastreamento_entregas`, `ecommerce_enderecos`  
**Branch:** `feat/squad2/kauan-leandro`  

Documentação técnica do notebook consolidado de Ingestão Silver (missões 02, 03, 04, 05 e 07 do estágio).

> **Nota:** Os notebooks individuais 08, 09, 10 e 11 foram unificados neste notebook e depois deletados.

## Visão geral

| Missão | Tabela | Regra | Resultado (1ª execução) |
| --- | --- | --- | --- |
| 02 | ecommerce_itens_pedido | FK com Silver de pedidos + tabela de espera | 34 Silver, 95 em espera |
| 03 | ecommerce_produtos | preco_lista > 0 e < 5000 + SKU 5-60 chars + is_ativo não nulo | 2832 Silver, 42 quarentena |
| 04 | ecommerce_categorias | Alertar se nº de categorias raiz mudar + id/nome não nulos | 135 Silver, 12 raízes registradas |
| 05 | ecommerce_rastreamento_entregas | status_entrega na lista de valores válidos | 25 Silver, 0 quarentena |
| 07 | ecommerce_enderecos | estado deve ser UF válida (2 letras) | 14 Silver, 0 quarentena |

## Estrutura do notebook unificado (16 células)

| Célula | Tipo | Função |
| --- | --- | --- |
| 1 | Markdown | Cabeçalho com Squad, integrantes, tabelas de escopo e objetivo |
| 2 | Python | `%run ./00_setup_config` — carrega credenciais .env |
| 3 | Python | Imports + config ADLS/UC + funções `tabela_existe()`, `merge_into()`, `append_quarentena()` + tabela `controle_categorias_raiz` |
| 4 | Markdown | Separador — Missão 02 (itens_pedido) |
| 5 | Python | **Missão 02:** Garante Silver de pedidos, retenta itens em espera, valida FK, MERGE INTO por `id_item_pedido` |
| 6 | Markdown | Separador — Missão 03 (produtos) |
| 7 | Python | **Missão 03:** Lê produtos, valida SKU/preco/is_ativo, MERGE INTO por `sku` |
| 8 | Markdown | Separador — Missão 04 (categorias) |
| 9 | Python | **Missão 04:** Lê categorias, valida id/nome, conta raízes, alerta se mudou, MERGE INTO por `id_categoria` |
| 10 | Markdown | Separador — Missão 05 (rastreamento) |
| 11 | Python | **Missão 05:** Lê rastreamento, valida status_entrega, MERGE INTO por `id_rastreamento` |
| 12 | Markdown | Separador — Missão 07 (enderecos) |
| 13 | Python | **Missão 07:** Lê enderecos, valida UF, MERGE INTO por `id_endereco` |
| 14 | Markdown | Resumo geral (markdown) |
| 15 | Python | Resumo executável com totais de lote, Silver, quarentena e alertas |
| 16 | Python | (vazio) |

## Arquitetura

Todos compartilham a mesma configuração (célula 3):
```
00_setup_config (credenciais .env)
    ↓
Teste de conectividade ADLS → se falhar, usa UC/Lakehouse Federation
    ↓
Para cada missão:
  Ler origem (sqlserver_catalog.squad2.*)
  → Filtro incremental (anti-join com Silver existente)
  → Aplicar regra de validação técnica + negócio
  → Válidos → Silver (MERGE INTO por chave de negócio, upsert idempotente)
  → Inválidos → Quarentena (append, mantém log de rejeições)
  → Otimização Delta: optimizeWrite + autoCompact
    ↓
Resumo geral (célula 12)
```

## Tabelas UC criadas

| Tabela | Função |
| --- | --- |
| `workspace.default.silver_ecommerce_pedidos` | Silver de pedidos (carga inicial da Missão 02) |
| `workspace.default.silver_ecommerce_itens_pedido` | Silver de itens_pedido (MERGE por `id_item_pedido`) |
| `workspace.default.itens_espera` | Itens órfãos sem pedido pai (MERGE por `id_item_pedido`) |
| `workspace.default.silver_ecommerce_produtos` | Silver de produtos (MERGE por `sku`) |
| `workspace.default.quarentena_ecommerce_produtos` | Produtos rejeitados (preço/SKU/is_ativo inválido) |
| `workspace.default.silver_ecommerce_categorias` | Silver de categorias (MERGE por `id_categoria`) |
| `workspace.default.quarentena_ecommerce_categorias` | Categorias rejeitadas (id/nome nulo) |
| `workspace.default.controle_categorias_raiz` | Controle de categorias raiz por lote |
| `workspace.default.silver_ecommerce_rastreamento` | Silver de rastreamento (MERGE por `id_rastreamento`) |
| `workspace.default.quarentena_ecommerce_rastreamento` | Rastreamento rejeitado (status inválido) |
| `workspace.default.silver_ecommerce_enderecos` | Silver de enderecos (MERGE por `id_endereco`) |
| `workspace.default.quarentena_ecommerce_enderecos` | Enderecos rejeitados (UF inválida) |

---
# Missão 03 — ecommerce_produtos (preco_lista)

## Regra
`preco_lista` deve ser > 0 e < 5000. Produtos fora do range vão para quarentena.

## Origem
`sqlserver_catalog.squad2.ecommerce_produtos` — 2874 linhas

## Colunas
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| sku | string | Identificador do produto |
| nome_produto | string | Nome do produto |
| descricao | string | Descrição |
| id_categoria | long | FK para ecommerce_categorias |
| preco_lista | double | Preço (validado) |
| unidade_medida | string | kg, L, etc. |
| nome_marca | string | Marca |
| is_ativo | boolean | Produto ativo? |

## Validação
- `SKU_NULO` — sku é NULL
- `SKU_TAMANHO_INVALIDO` — comprimento do SKU fora de 5-60 caracteres
- `IS_ATIVO_NULO` — is_ativo é NULL
- `PRECO_NULO` — preco_lista é NULL
- `PRECO_MENOR_OU_IGUAL_ZERO` — preco_lista ≤ 0
- `PRECO_MAIOR_OU_IGUAL_5000` — preco_lista ≥ 5000

## Persistência
- Válidos → `workspace.default.silver_ecommerce_produtos` (MERGE INTO por `sku`, idempotente)
- Rejeitados → `workspace.default.quarentena_ecommerce_produtos` (append, mantém log de rejeições)

## Resultado
- 2832 válidos → `workspace.default.silver_ecommerce_produtos`
- 42 rejeitados → `workspace.default.quarentena_ecommerce_produtos`
  - 41 com PRECO_MENOR_OU_IGUAL_ZERO
  - 1 com PRECO_MAIOR_OU_IGUAL_5000
- preco_lista min: -48.84, max: 7048.0

---
# 09_ingestao_silver_categorias — Missão 04

## Objetivo
Alertar se o número total de categorias raiz (`id_categoria_pai IS NULL`) mudar entre lotes. Não bloqueia o processamento — apenas emite alerta. Também valida que `id_categoria` e `nome_categoria` não são nulos nem vazios.

## Origem
`sqlserver_catalog.squad2.ecommerce_categorias` — 135 linhas

## Colunas da tabela
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| id_categoria | long | ID da categoria |
| nome_categoria | string | Nome |
| id_categoria_pai | double | ID da categoria pai (NULL = raiz) |
| nome_categoria_pai | string | Nome da categoria pai |
| tipo_categoria | string | "Categoria" (raiz) ou "Subcategoria" |

## Células

| Célula | Título | Função |
| --- | --- | --- |
| 1 | Markdown | Título e descrição da regra |
| 2 | Setup | `%run ./00_setup_config` |
| 3 | Importacao | Importa bibliotecas |
| 4 | Configuracao camadas | ADLS + UC fallback. Define tabela de controle `controle_categorias_raiz` |
| 5 | Criar tabela controle | Cria `workspace.default.controle_categorias_raiz` (total, nomes, timestamp) |
| 6 | Ler categorias origem | Lê `ecommerce_categorias` via Lakehouse Federation. Anti-join por `id_categoria` |
| 7 | Validar categorias raiz | Conta categorias com `id_categoria_pai IS NULL`. Compara com lote anterior (tabela de controle). Se diferente → alerta com detalhes (adicionadas/removidas) |
| 8 | Gravar Silver e controle | Append categorias na Silver. INSERT na tabela de controle com total atual |
| 9 | Resumo | Imprime totais, status do alerta e histórico de controle |

## Validação
- `ID_CATEGORIA_NULO` — id_categoria é NULL
- `NOME_CATEGORIA_NULO` — nome_categoria é NULL ou vazio

## Persistência
- Válidos → `workspace.default.silver_ecommerce_categorias` (MERGE INTO por `id_categoria`, idempotente)
- Rejeitados → `workspace.default.quarentena_ecommerce_categorias` (append, mantém log de rejeições)

## Tabela de controle
```sql
CREATE TABLE workspace.default.controle_categorias_raiz (
    total_categorias_raiz  BIGINT,
    nomes_categorias_raiz STRING,
    dt_processamento       TIMESTAMP
) USING DELTA
```

## Resultado da execução
- 135 categorias na origem (todas novas — primeira carga)
- 12 categorias raiz identificadas:
  - Adega & Destilados, Bazar & Utilidades, Bebidas, Higiene & Cuidados Pessoais, Hortifruti Seco & Castanhas, Limpeza, Matinais Doces & Biscoitos, Mercearia Básica, Mercearia Salgada, Mundo Bebê, Padaria Industrializada, Pets
- 135 categorias gravadas na Silver
- Controle registrado: 12 raízes
- Sem alerta (primeira execução)

---
# 10_ingestao_silver_rastreamento — Missão 05

## Objetivo
Validar que `status_entrega` está na lista de valores do fluxo logístico. Registros com status fora do fluxo vão para quarentena.

## Origem
`sqlserver_catalog.squad2.ecommerce_rastreamento_entregas` — 25 linhas

## Colunas da tabela
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| id_rastreamento | long | ID do rastreamento |
| id_pedido_ecommerce | long | FK para ecommerce_pedidos |
| codigo_rastreio | string | Código de rastreio |
| id_transportadora | long | ID da transportadora |
| nome_transportadora | string | Nome da transportadora |
| status_entrega | string | Status (validado) |
| dt_evento | timestamp | Data do evento |
| observacao | string | Observação |
| bronze_source_file | string | Arquivo de origem |

## Lista de status válidos (confirmar com o professor)
```
aguardando separacao, em separacao, separado, em transito, entregue, falha, devolvido
+ variações com underscore: aguardando_separacao, em_separacao, em_transito
```

## Células

| Célula | Título | Função |
| --- | --- | --- |
| 1 | Markdown | Título e descrição da regra |
| 2 | Setup | `%run ./00_setup_config` |
| 3 | Importacao | Importa bibliotecas |
| 4 | Configuracao camadas | ADLS + UC fallback |
| 5 | Parametros status validos | Define `STATUS_VALIDOS` (lista de 10 valores) e função `tabela_existe()` |
| 6 | Ler rastreamento origem | Lê `ecommerce_rastreamento_entregas` via Lakehouse Federation. Anti-join por `id_rastreamento` |
| 7 | Validar status_entrega | Normaliza status (trim + lowercase). Classifica: `STATUS_NULO`, `STATUS_FORA_DO_FLOW` |
| 8 | Montar Silver e quarentena | Adiciona `silver_processed_at`. Separa válidos de inválidos |
| 9 | Gravar Silver e quarentena | Append em `silver_ecommerce_rastreamento` e `quarentena_ecommerce_rastreamento` |
| 10 | Resumo | Imprime totais, distribuição de quarentena e amostra da Silver |

## Resultado da execução
- 25 rastreamentos na origem (todos novos — primeira carga)
- 25 válidos → `workspace.default.silver_ecommerce_rastreamento`
- 0 rejeitados → todos com status "em separacao" (válido)
- Transportadoras: Correios, JadLog, Loggi, Sequoia

---
# 11_ingestao_silver_enderecos — Missão 07

## Objetivo
Validar que `estado` é uma UF válida (2 letras maiúsculas, correspondente a um estado brasileiro). Enderecos com UF inválida vão para quarentena.

## Origem
`sqlserver_catalog.squad2.ecommerce_enderecos` — 14 linhas

## Colunas da tabela
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| id_endereco | long | ID do endereço |
| id_cliente | long | FK para ecommerce_clientes |
| logradouro | string | Rua/avenida |
| numero | long | Número |
| complemento | string | Complemento |
| bairro | string | Bairro |
| cep | string | CEP |
| cidade | string | Cidade |
| estado | string | UF (validado) |
| latitude | double | Latitude |
| longitude | double | Longitude |
| apelido | string | Apelido (Casa, Trabalho) |
| is_principal | boolean | Endereço principal? |
| bronze_source_file | string | Arquivo de origem |

## Lista de UFs válidas (26 estados + DF)
```
AC, AP, AM, BA, CE, DF, ES, GO, MA, MT, MS, MG, PA, PB, PR, PE,
PI, RJ, RN, RS, RO, RR, SC, SP, SE, TO
```

## Células

| Célula | Título | Função |
| --- | --- | --- |
| 1 | Markdown | Título e descrição da regra |
| 2 | Setup | `%run ./00_setup_config` |
| 3 | Importacao | Importa bibliotecas |
| 4 | Configuracao camadas | ADLS + UC fallback |
| 5 | Parametros UF validas | Define `UFS_VALIDAS` (26 UFs) e função `tabela_existe()` |
| 6 | Ler enderecos origem | Lê `ecommerce_enderecos` via Lakehouse Federation. Anti-join por `id_endereco` |
| 7 | Validar UF | Normaliza estado (trim + uppercase). Classifica: `ESTADO_NULO`, `UF_FORMATO_INVALIDO`, `UF_NAO_RECONHECIDA` |
| 8 | Montar Silver e quarentena | Adiciona `silver_processed_at`. Separa válidos de inválidos |
| 9 | Gravar Silver e quarentena | Append em `silver_ecommerce_enderecos` e `quarentena_ecommerce_enderecos` |
| 10 | Resumo | Imprime totais, distribuição de quarentena e amostra da Silver |

## Resultado da execução
- 14 enderecos na origem (todos novos — primeira carga)
- 14 válidos → `workspace.default.silver_ecommerce_enderecos`
- 0 rejeitados → todos com UF válida
- Distribuição de UFs: SP=6, MG=4, ES=2, RJ=2

---
# Resumo geral — Status das 7 missões

| # | Tabela | Regra | Notebook | Status | Resultado |
| --- | --- | --- | --- | --- | --- |
| 01 | ecommerce_pedidos | Controle de hash (não reprocessar) | 04_ingestao_bronze (unificado) | ✅ Executado | 90343 Silver, hash controle |
| 02 | ecommerce_itens_pedido | FK com Silver de pedidos + espera | 08_ingestao_silver_all (unificado) | ✅ Executado | 34 Silver, 95 em espera |
| 03 | ecommerce_produtos | SKU 5-60 + preco > 0 e < 5000 + is_ativo | 08_ingestao_silver_all (unificado) | ✅ Executado | 2832 Silver, 42 quarentena |
| 04 | ecommerce_categorias | id/nome não nulos + alerta de raiz | 08_ingestao_silver_all (unificado) | ✅ Executado | 135 Silver, 12 raízes |
| 05 | ecommerce_rastreamento_entregas | status_entrega válido | 08_ingestao_silver_all (unificado) | ✅ Executado | 25 Silver, 0 quarentena |
| 06 | ecommerce_clientes | Email único no lote e Silver | 05_ingestao_silver (Kauan) | ✅ Executado | 361 Silver, 12 quarentena |
| 07 | ecommerce_enderecos | UF válida (2 letras) | 08_ingestao_silver_all (unificado) | ✅ Executado | 14 Silver, 0 quarentena |

## Pontos resolvidos
1. **Schema de ecommerce_itens_pedido**: definido como `sqlserver_catalog.squad1.ecommerce_itens_pedido` (129 linhas em squad1).
2. **Lista de status do fluxo logístico** (missão 05): definida com 10 valores válidos (incluindo variações com underscore).
3. **Silver de pedidos** (missão 02): carga inicial criada automaticamente no notebook 08 (90343 registros).
4. **MERGE INTO**: todas as missões persistem via MERGE INTO por chave de negócio (idempotente).
5. **Ordem de execução**: 04 (Bronze unificado: pedidos + clientes) → 08_ingestao_silver_all (5 missões) → 05 (Silver clientes)